In [1]:
import pandas as pd
import numpy as np
import os

In [4]:
train = pd.read_csv("artifacts/train.csv")
validation = pd.read_csv("artifacts/validation.csv")
test = pd.read_csv("artifacts/test.csv")

print("Train shape:", train.shape)
print("Validation shape:", validation.shape)
print("Test shape:", test.shape)

Train shape: (69608, 33)
Validation shape: (14916, 33)
Test shape: (14917, 33)


In [5]:
print("Columns:")
for i, col in enumerate(train.columns, start=1):
    print(f"{i}. {col}")

Columns:
1. order_id
2. customer_id
3. order_status
4. order_purchase_timestamp
5. order_approved_at
6. order_delivered_carrier_date
7. order_delivered_customer_date
8. order_estimated_delivery_date
9. customer_unique_id
10. customer_zip_code_prefix
11. customer_city
12. customer_state
13. geolocation_zip_code_prefix
14. latitude
15. longitude
16. city
17. state
18. item_count
19. unique_products
20. unique_sellers
21. total_item_price
22. mean_item_price
23. max_item_price
24. total_freight_value
25. mean_freight_value
26. mean_product_weight_g
27. product_category_count
28. seller_state_count
29. payment_count
30. total_payment_value
31. mean_payment_value
32. max_payment_installments
33. is_late


In [6]:
print("\nTarget:", "is_late" in train.columns)
print("Train shape:", train.shape)


Target: True
Train shape: (69608, 33)


In [7]:
# Columns that must not be used as model features

columns_to_drop = [
    "is_late",                         # Target
    "order_id",                        # Order identifier
    "customer_id",                     # Customer identifier
    "customer_unique_id",              # Customer identifier
    "order_status",                    # Potential outcome/future information
    "order_delivered_carrier_date",    # Happens after prediction time
    "order_delivered_customer_date"    # Used to determine the target
]

X_train = train.drop(columns=columns_to_drop)
y_train = train["is_late"].copy()

X_validation = validation.drop(columns=columns_to_drop)
y_validation = validation["is_late"].copy()

X_test = test.drop(columns=columns_to_drop)
y_test = test["is_late"].copy()

print("X_train shape:", X_train.shape)
print("X_validation shape:", X_validation.shape)
print("X_test shape:", X_test.shape)

print("\nTarget:")
print(y_train.value_counts(normalize=True))

X_train shape: (69608, 26)
X_validation shape: (14916, 26)
X_test shape: (14917, 26)

Target:
is_late
0    0.921288
1    0.078712
Name: proportion, dtype: float64


In [8]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_estimated_delivery_date"
]

for df in [X_train, X_validation, X_test]:
    for col in date_columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

print(X_train[date_columns].dtypes)

print("\nMissing values in training dates:")
print(X_train[date_columns].isna().sum())

C:\Users\Rowaida\AppData\Local\Temp\ipykernel_2592\554582963.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors="coerce")


order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

Missing values in training dates:
order_purchase_timestamp           0
order_approved_at                124
order_estimated_delivery_date      0
dtype: int64


In [9]:
def create_date_features(df):
    df = df.copy()

    # Features from purchase date
    df["purchase_year"] = df["order_purchase_timestamp"].dt.year
    df["purchase_month"] = df["order_purchase_timestamp"].dt.month
    df["purchase_dayofweek"] = df["order_purchase_timestamp"].dt.dayofweek
    df["purchase_hour"] = df["order_purchase_timestamp"].dt.hour

    # Feature from estimated delivery date
    df["estimated_delivery_month"] = (
        df["order_estimated_delivery_date"].dt.month
    )

    # Time available until the estimated delivery date
    df["estimated_delivery_window_days"] = (
        df["order_estimated_delivery_date"]
        - df["order_purchase_timestamp"]
    ).dt.total_seconds() / (60 * 60 * 24)

    # Time between purchase and approval
    df["approval_delay_hours"] = (
        df["order_approved_at"]
        - df["order_purchase_timestamp"]
    ).dt.total_seconds() / 3600

    # Remove the original datetime columns
    df = df.drop(columns=date_columns)

    return df


X_train = create_date_features(X_train)
X_validation = create_date_features(X_validation)
X_test = create_date_features(X_test)

print("X_train shape:", X_train.shape)
print("X_validation shape:", X_validation.shape)
print("X_test shape:", X_test.shape)

print("\nRemaining datetime columns:")
print(X_train.select_dtypes(include=["datetime64"]).columns.tolist())

X_train shape: (69608, 30)
X_validation shape: (14916, 30)
X_test shape: (14917, 30)

Remaining datetime columns:
[]


In [10]:
categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

print("Categorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numerical_features)

print("\nNumber of categorical features:", len(categorical_features))
print("Number of numerical features:", len(numerical_features))

Categorical features:
['customer_city', 'customer_state', 'city', 'state']

Numerical features:
['customer_zip_code_prefix', 'geolocation_zip_code_prefix', 'latitude', 'longitude', 'item_count', 'unique_products', 'unique_sellers', 'total_item_price', 'mean_item_price', 'max_item_price', 'total_freight_value', 'mean_freight_value', 'mean_product_weight_g', 'product_category_count', 'seller_state_count', 'payment_count', 'total_payment_value', 'mean_payment_value', 'max_payment_installments', 'purchase_year', 'purchase_month', 'purchase_dayofweek', 'purchase_hour', 'estimated_delivery_month', 'estimated_delivery_window_days', 'approval_delay_hours']

Number of categorical features: 4
Number of numerical features: 26


In [11]:
categorical_cardinality = pd.DataFrame({
    "unique_values": X_train[categorical_features].nunique(),
    "missing_values": X_train[categorical_features].isna().sum()
})

categorical_cardinality

,unique_values,missing_values
customer_city,3702,0
customer_state,27,0
city,3798,206
state,27,206


In [12]:
print("Customer ZIP unique values:",
      X_train["customer_zip_code_prefix"].nunique())

print("Geolocation ZIP unique values:",
      X_train["geolocation_zip_code_prefix"].nunique())

Customer ZIP unique values: 13839
Geolocation ZIP unique values: 13712


In [13]:
high_cardinality_location_cols = [
    "customer_city",
    "city",
    "customer_zip_code_prefix",
    "geolocation_zip_code_prefix"
]

X_train = X_train.drop(columns=high_cardinality_location_cols)
X_validation = X_validation.drop(columns=high_cardinality_location_cols)
X_test = X_test.drop(columns=high_cardinality_location_cols)

print("X_train shape:", X_train.shape)
print("X_validation shape:", X_validation.shape)
print("X_test shape:", X_test.shape)

X_train shape: (69608, 26)
X_validation shape: (14916, 26)
X_test shape: (14917, 26)


In [14]:
categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

print("Categorical features:", categorical_features)
print("Numerical features:", numerical_features)

print("\nNumber of categorical features:", len(categorical_features))
print("Number of numerical features:", len(numerical_features))
print("Total features:", len(categorical_features) + len(numerical_features))

Categorical features: ['customer_state', 'state']
Numerical features: ['latitude', 'longitude', 'item_count', 'unique_products', 'unique_sellers', 'total_item_price', 'mean_item_price', 'max_item_price', 'total_freight_value', 'mean_freight_value', 'mean_product_weight_g', 'product_category_count', 'seller_state_count', 'payment_count', 'total_payment_value', 'mean_payment_value', 'max_payment_installments', 'purchase_year', 'purchase_month', 'purchase_dayofweek', 'purchase_hour', 'estimated_delivery_month', 'estimated_delivery_window_days', 'approval_delay_hours']

Number of categorical features: 2
Number of numerical features: 24
Total features: 26


In [15]:
missing_summary = X_train.isnull().sum()

missing_summary = missing_summary[
    missing_summary > 0
].sort_values(ascending=False)

print("Missing values in X_train:")
print(missing_summary)

Missing values in X_train:
mean_product_weight_g       567
item_count                  559
unique_sellers              559
mean_freight_value          559
product_category_count      559
seller_state_count          559
unique_products             559
mean_item_price             559
max_item_price              559
total_freight_value         559
total_item_price            559
state                       206
longitude                   206
latitude                    206
approval_delay_hours        124
payment_count                 1
total_payment_value           1
mean_payment_value            1
max_payment_installments      1
dtype: int64


In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [17]:
# Numerical preprocessing
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

# Categorical preprocessing
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

# Combine numerical and categorical preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


In [18]:
# Fit ONLY on the training features
X_train_processed = preprocessor.fit_transform(X_train)

# Apply the already-fitted preprocessor to validation and test
X_validation_processed = preprocessor.transform(X_validation)
X_test_processed = preprocessor.transform(X_test)

print("Processed train shape:", X_train_processed.shape)
print("Processed validation shape:", X_validation_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Processed train shape: (69608, 78)
Processed validation shape: (14916, 78)
Processed test shape: (14917, 78)


In [19]:
feature_names = preprocessor.get_feature_names_out().tolist()

print("Number of final features:", len(feature_names))
print("\nFirst 20 feature names:")
print(feature_names[:20])

Number of final features: 78

First 20 feature names:
['num__latitude', 'num__longitude', 'num__item_count', 'num__unique_products', 'num__unique_sellers', 'num__total_item_price', 'num__mean_item_price', 'num__max_item_price', 'num__total_freight_value', 'num__mean_freight_value', 'num__mean_product_weight_g', 'num__product_category_count', 'num__seller_state_count', 'num__payment_count', 'num__total_payment_value', 'num__mean_payment_value', 'num__max_payment_installments', 'num__purchase_year', 'num__purchase_month', 'num__purchase_dayofweek']


In [20]:
from pathlib import Path
import joblib
import json

# Create the artifacts folder if it doesn't exist
feature_artifact_dir = Path("artifacts/features")
feature_artifact_dir.mkdir(parents=True, exist_ok=True)

# Save the fitted preprocessor
joblib.dump(
    preprocessor,
    feature_artifact_dir / "preprocessor.joblib"
)

# Save the final feature names
with open(
    feature_artifact_dir / "feature_names.json",
    "w"
) as f:
    json.dump(feature_names, f, indent=2)

print("Saved artifacts:")
print(feature_artifact_dir / "preprocessor.joblib")
print(feature_artifact_dir / "feature_names.json")

Saved artifacts:
artifacts\features\preprocessor.joblib
artifacts\features\feature_names.json


In [22]:
import numpy as np

# Save processed feature matrices
np.save(
    feature_artifact_dir / "X_train_processed.npy",
    X_train_processed
)

np.save(
    feature_artifact_dir / "X_validation_processed.npy",
    X_validation_processed
)

np.save(
    feature_artifact_dir / "X_test_processed.npy",
    X_test_processed
)

# Save target arrays
np.save(
    feature_artifact_dir / "y_train.npy",
    y_train.to_numpy()
)

np.save(
    feature_artifact_dir / "y_validation.npy",
    y_validation.to_numpy()
)

np.save(
    feature_artifact_dir / "y_test.npy",
    y_test.to_numpy()
)

print("All processed datasets and targets saved successfully.")

All processed datasets and targets saved successfully.


In [24]:
print("Final Notebook 5 verification")
print("=" * 40)

print("Processed train:", X_train_processed.shape)
print("Processed validation:", X_validation_processed.shape)
print("Processed test:", X_test_processed.shape)

print("\nTargets:")
print("y_train:", y_train.shape)
print("y_validation:", y_validation.shape)
print("y_test:", y_test.shape)

print("\nArtifacts:")
for file in sorted(feature_artifact_dir.iterdir()):
    print("-", file.name)

Final Notebook 5 verification
Processed train: (69608, 78)
Processed validation: (14916, 78)
Processed test: (14917, 78)

Targets:
y_train: (69608,)
y_validation: (14916,)
y_test: (14917,)

Artifacts:
- feature_names.json
- preprocessor.joblib
- X_test_processed.npy
- X_train_processed.npy
- X_validation_processed.npy
- y_test.npy
- y_train.npy
- y_validation.npy


In [25]:
feature_manifest = {
    "n_input_features": 26,
    "n_output_features": len(feature_names),
    "categorical_features": categorical_features,
    "numerical_features": numerical_features,
    "dropped_high_cardinality_features": [
        "customer_city",
        "city",
        "customer_zip_code_prefix",
        "geolocation_zip_code_prefix"
    ],
    "date_features_created": [
        "purchase_year",
        "purchase_month",
        "purchase_dayofweek",
        "purchase_hour",
        "estimated_delivery_month",
        "estimated_delivery_window_days",
        "approval_delay_hours"
    ],
    "numeric_imputation": "median",
    "numeric_scaling": "StandardScaler",
    "categorical_imputation": "most_frequent",
    "categorical_encoding": "OneHotEncoder(handle_unknown='ignore')"
}

with open(
    feature_artifact_dir / "feature_manifest.json",
    "w"
) as f:
    json.dump(feature_manifest, f, indent=2)

print("Feature manifest saved successfully.")

Feature manifest saved successfully.
